# Analisis Exploratorio - Forecasting de Demanda

Este notebook realiza un analisis exploratorio completo de los datos de ventas antes de entrenar los modelos.

**Contenido:**
1. Carga y descripcion de datos
2. Visualizacion de series de tiempo
3. Analisis de estacionalidad (semanal y mensual)
4. Test de estacionariedad (Dickey-Fuller)
5. Descomposicion de la serie
6. Correlacion entre productos
7. Deteccion de outliers

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)

print('Librerias cargadas correctamente')

## 1. Carga y descripcion de datos

In [ ]:
import os

# Generar datos si no existen
DATA_PATH = '../data/sales_data.csv'
if not os.path.exists(DATA_PATH):
    from data.generate_data import generate_sales_data
    df_raw = generate_sales_data()
    df_raw.to_csv(DATA_PATH, index=False)
    print(f'Datos generados: {len(df_raw)} filas')

df = pd.read_csv(DATA_PATH, parse_dates=['fecha'])
print(f'Shape: {df.shape}')
print(f'Rango de fechas: {df.fecha.min()} -> {df.fecha.max()}')
print(f'Productos: {df.producto.unique().tolist()}')
df.head(10)

In [ ]:
# Estadisticas descriptivas
print('=== Estadisticas descriptivas por producto ===')
df.groupby('producto')['ventas'].describe().round(2)

In [ ]:
# Valores faltantes
missing = df.isnull().sum()
print('Valores faltantes:')
print(missing[missing > 0])
print(f'\nPorcentaje faltantes en ventas: {df.ventas.isna().mean()*100:.2f}%')

## 2. Visualizacion de series de tiempo

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for ax, product in zip(axes, df.producto.unique()):
    data = df[df.producto == product].copy()
    ax.plot(data.fecha, data.ventas, linewidth=0.8, alpha=0.7, color='steelblue', label='Ventas diarias')
    ma7 = data.ventas.rolling(7, center=True).mean()
    ax.plot(data.fecha, ma7, linewidth=2, color='red', label='Media movil 7d')
    ax.set_title(f'{product}', fontweight='bold')
    ax.set_ylabel('Unidades')
    ax.legend(loc='upper left')

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.suptitle('Serie Temporal de Ventas por Producto', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Analisis de estacionalidad

In [ ]:
product = 'Producto_A'
data = df[df.producto == product].copy()
data['dia_semana'] = data.fecha.dt.day_name()
data['mes'] = data.fecha.dt.month_name()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Estacionalidad semanal
orden_dias = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
semanal = data.groupby('dia_semana')['ventas'].mean().reindex(orden_dias)
axes[0].bar(range(7), semanal.values, color=sns.color_palette('Set2', 7), edgecolor='grey')
axes[0].set_xticks(range(7))
axes[0].set_xticklabels(['Lun','Mar','Mie','Jue','Vie','Sab','Dom'])
axes[0].set_title(f'Promedio de ventas por dia - {product}', fontweight='bold')
axes[0].set_ylabel('Ventas promedio')

# Estacionalidad mensual
data['mes_num'] = data.fecha.dt.month
mensual = data.groupby('mes_num')['ventas'].mean()
axes[1].plot(mensual.index, mensual.values, marker='o', linewidth=2, color='tomato')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'])
axes[1].set_title(f'Promedio de ventas por mes - {product}', fontweight='bold')
axes[1].set_ylabel('Ventas promedio')

plt.tight_layout()
plt.show()

## 4. Test de Estacionariedad (Dickey-Fuller)

In [ ]:
print('=== Test de Dickey-Fuller Aumentado ===')
print('H0: La serie NO es estacionaria (tiene raiz unitaria)')
print('Si p-valor < 0.05 -> rechazamos H0 -> la serie ES estacionaria\n')

for product in df.producto.unique():
    series = (
        df[df.producto == product]
        .set_index('fecha')['ventas']
        .sort_index()
        .asfreq('D')
        .ffill()
    )
    result = adfuller(series.dropna(), autolag='AIC')
    p_val = result[1]
    es_estacionaria = 'SI' if p_val < 0.05 else 'NO'
    print(f'{product}: p-valor={p_val:.4f} | Estacionaria: {es_estacionaria}')
    if p_val >= 0.05:
        print(f'  -> Necesita diferenciacion (d=1 en ARIMA)')

## 5. Descomposicion de la Serie

In [ ]:
product = 'Producto_A'
series = (
    df[df.producto == product]
    .set_index('fecha')['ventas']
    .sort_index()
    .asfreq('D')
    .ffill()
)

decomp = seasonal_decompose(series, model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
labels = ['Serie original', 'Tendencia', 'Estacionalidad (periodo=7)', 'Residuo']
colors = ['steelblue', 'darkorange', 'seagreen', 'gray']
components = [series, decomp.trend, decomp.seasonal, decomp.resid]

for ax, data, label, color in zip(axes, components, labels, colors):
    ax.plot(data.index, data.values, color=color, linewidth=0.9)
    ax.set_ylabel(label, fontsize=10)

axes[0].set_title(f'Descomposicion Estacional - {product}', fontweight='bold', fontsize=13)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f'Fuerza de la tendencia: {1 - decomp.resid.var() / (decomp.trend.dropna() + decomp.resid.dropna()).var():.3f}')
print(f'Fuerza de la estacionalidad: {1 - decomp.resid.var() / (decomp.seasonal + decomp.resid.dropna()).var():.3f}')

## 6. Correlacion entre productos

In [ ]:
pivot = df.pivot_table(index='fecha', columns='producto', values='ventas')
corr = pivot.corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlacion de Ventas entre Productos', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Deteccion de Outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot por producto
df.boxplot(column='ventas', by='producto', ax=axes[0])
axes[0].set_title('Distribucion de Ventas por Producto')
axes[0].set_xlabel('Producto')
axes[0].set_ylabel('Ventas')
plt.sca(axes[0])
plt.title('Distribucion de Ventas por Producto')

# Histograma
for product in df.producto.unique():
    data = df[df.producto == product]['ventas'].dropna()
    axes[1].hist(data, bins=30, alpha=0.6, label=product, edgecolor='white')
axes[1].set_title('Distribucion de Frecuencias')
axes[1].set_xlabel('Ventas')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.show()

# Calcular outliers con IQR
print('=== Outliers detectados (metodo IQR, multiplicador=3) ===')
for product in df.producto.unique():
    data = df[df.producto == product]['ventas'].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    outliers = data[(data < Q1 - 3*IQR) | (data > Q3 + 3*IQR)]
    print(f'{product}: {len(outliers)} outliers | rango normal [{Q1-3*IQR:.0f}, {Q3+3*IQR:.0f}]')

## Siguiente Paso

Con el analisis exploratorio completado, ejecutar el pipeline completo:

```bash
python main.py
```

Esto entrena SARIMA, Prophet y XGBoost, compara resultados con MAE/RMSE y genera todas las graficas.